# Long audio -> 46 s clips (music removed, starts where speech starts)

1. Runtime > Change runtime type > **T4 GPU** (much faster)
2. Run cell 1 (install)
3. Run cell 2 and pick your **original** recording(s) (not a file this notebook already made). For each one it removes the background music, finds where the person starts speaking, then cuts the rest into back-to-back **46.0 s** clips (mono, 44.1 kHz wav, `urdu_001.wav`, `urdu_002.wav`, ...). A leftover piece shorter than 46 s at the end is dropped.
4. Everything downloads to your PC as `clips.zip`.
5. Unzip `clips.zip` into the **`audios`** folder of the repo, then continue with `python transcribe_audios.py` (see the README).


In [ ]:
!pip install -q demucs silero-vad

In [ ]:
import os, shutil, subprocess, sys, wave
import numpy as np
import torch
from google.colab import files
from silero_vad import load_silero_vad, get_speech_timestamps

# ====== SETTINGS ======
CLIP_SEC = 46        # length of every clip in seconds
PAD_BEFORE = 0.2     # start this many seconds before the first word so it isn't clipped
PREFIX = "urdu"      # clips are named urdu_001.wav, urdu_002.wav, ...
START_INDEX = 1      # number of the first clip (change it if you already have earlier ones)
# ======================

SR = 44100

def run(cmd):
    subprocess.run(cmd, check=True)

vad_model = load_silero_vad()
shutil.rmtree("clips", ignore_errors=True)
os.makedirs("clips")

uploaded = files.upload()          # pick one or more original recordings
n = START_INDEX

for src in uploaded:
    print(f"\n== {src}")
    shutil.rmtree("separated", ignore_errors=True)

    # 1. Remove background music from the whole recording (keeps the voice only)
    run(["ffmpeg", "-y", "-loglevel", "error", "-i", src, "-vn", "-ac", "2", "-ar", str(SR), "input.wav"])
    run([sys.executable, "-m", "demucs", "--two-stems=vocals", "--mp3", "--mp3-bitrate=320",
         "-n", "htdemucs", "-o", "separated", "input.wav"])
    run(["ffmpeg", "-y", "-loglevel", "error", "-i", "separated/htdemucs/input/vocals.mp3",
         "-ac", "1", "-ar", str(SR), "vocals.wav"])
    with wave.open("vocals.wav") as w:
        audio = np.frombuffer(w.readframes(w.getnframes()), dtype=np.int16)

    # 2. Find where the person starts speaking
    raw = subprocess.run(["ffmpeg", "-loglevel", "error", "-i", "vocals.wav", "-ac", "1", "-ar", "16000",
                          "-f", "s16le", "-"], check=True, capture_output=True).stdout
    wav16 = torch.from_numpy(np.frombuffer(raw, dtype=np.int16).astype(np.float32) / 32768.0)
    speech = get_speech_timestamps(wav16, vad_model, sampling_rate=16000,
                                   return_seconds=True, min_speech_duration_ms=500)
    if not speech:
        print("  No speech found, skipping this file.")
        continue
    start = max(0.0, speech[0]["start"] - PAD_BEFORE)
    print(f"  Speech starts at {start:.1f}s, file is {len(audio) / SR:.1f}s long")

    # 3. Cut back-to-back clips of exactly CLIP_SEC seconds
    size = int(CLIP_SEC * SR)
    pos = int(start * SR)
    count = 0
    while pos + size <= len(audio):
        with wave.open(f"clips/{PREFIX}_{n:03d}.wav", "wb") as w:
            w.setnchannels(1)
            w.setsampwidth(2)
            w.setframerate(SR)
            w.writeframes(audio[pos:pos + size].tobytes())
        pos += size
        n += 1
        count += 1
    print(f"  {count} clips, dropped {(len(audio) - pos) / SR:.1f}s leftover at the end")

print(f"\nDone: {n - START_INDEX} clips")
shutil.make_archive("clips", "zip", "clips")
files.download("clips.zip")

In [ ]:
import os
import glob
import shutil

# 1. Remove temporary directories
for folder in ["clips", "separated"]:
    if os.path.exists(folder):
        shutil.rmtree(folder)
        print(f"Removed folder: {folder}")

# 2. Remove standard intermediate files
for file in ["vocals.wav", "input.wav", "clips.zip"]:
    if os.path.exists(file):
        os.remove(file)
        print(f"Removed file: {file}")

# 3. Remove any uploaded audio files (mp3, wav, etc.) from the workspace
for ext in ["*.mp3", "*.wav", "*.ogg", "*.m4a", "*.flac"]:
    for file in glob.glob(ext):
        # Don't delete the output files if they were recreated in the current directory
        if file not in ["vocals.wav", "input.wav"]:
            os.remove(file)
            print(f"Removed uploaded file: {file}")

print("Workspace cleaned! You are ready to upload new audio files.")